In [1]:
!pip install timm librosa albumentations -q

In [26]:
import os, gc, random, warnings, time
from pathlib import Path
 
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import timm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
 
warnings.filterwarnings("ignore")

In [27]:
class BaseCFG:
    BASE_DIR       = Path("/kaggle/input/competitions/birdclef-2026")
    TRAIN_CSV      = BASE_DIR / "train.csv"
    TAXON_CSV      = BASE_DIR / "taxonomy.csv"
    LABELS_CSV     = BASE_DIR / "train_soundscapes_labels.csv"
    AUDIO_DIR      = BASE_DIR / "train_audio"
    SOUNDSCAPE_DIR = BASE_DIR / "train_soundscapes"
    OUTPUT_DIR     = Path("/kaggle/working")

    SR             = 32_000
    DURATION       = 5
    N_FFT          = 1024
    HOP_LENGTH     = 512
    N_MELS         = 128
    FMIN           = 50
    FMAX           = 14_000

    SEED           = 42
    FOLDS          = 5
    TRAIN_FOLD     = 0
    EPOCHS         = 10
    BATCH_SIZE     = 32
    LR             = 1e-3
    WEIGHT_DECAY   = 1e-4
    NUM_WORKERS    = 0
    AMP            = True
    PRETRAINED     = True
    DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"

    MODEL          = "efficientnet_b0"
    TAG            = "approach"      
    MIXUP_ALPHA    = 0.0
    SPEC_AUG       = False
    FREQ_MASK      = 20
    TIME_MASK      = 40
    NUM_MASKS      = 2
    USE_PCEN       = False

In [29]:
def seed_everything(cfg):
    random.seed(cfg.SEED); np.random.seed(cfg.SEED)
    torch.manual_seed(cfg.SEED); torch.cuda.manual_seed_all(cfg.SEED)
    os.environ["PYTHONHASHSEED"] = str(cfg.SEED)

Data Loading:

In [30]:
def load_and_prepare(cfg):
    train    = pd.read_csv(cfg.TRAIN_CSV)
    taxonomy = pd.read_csv(cfg.TAXON_CSV)
    labels   = pd.read_csv(cfg.LABELS_CSV)

    all_species = sorted(taxonomy["primary_label"].tolist())
    sp2idx      = {sp: i for i, sp in enumerate(all_species)}
    num_classes = len(all_species)

    def encode_labels(row):
        vec = np.zeros(num_classes, dtype=np.float32)
        for sp in str(row["primary_label"]).split(";"):
            if sp in sp2idx:
                vec[sp2idx[sp]] = 1.0
        sec = str(row.get("secondary_labels", "")).strip("[]").replace("'", "")
        for sp in sec.split(", "):
            if sp.strip() in sp2idx:
                vec[sp2idx[sp.strip()]] = 0.5
        return vec

    train["label_vec"] = train.apply(encode_labels, axis=1)
    train["filepath"]  = train["filename"].apply(lambda f: str(cfg.AUDIO_DIR / f))

    sc_rows = []
    for _, r in labels.iterrows():
        vec = np.zeros(num_classes, dtype=np.float32)
        for sp in str(r["primary_label"]).split(";"):
            sp = sp.strip()
            if sp in sp2idx:
                vec[sp2idx[sp]] = 1.0
        fpath = cfg.SOUNDSCAPE_DIR / r["filename"]
        if fpath.exists():
            sc_rows.append({
                "filepath":      str(fpath),
                "start":         r["start"],
                "label_vec":     vec,
                "primary_label": r["primary_label"],
            })
    sc_df = pd.DataFrame(sc_rows)

    skf = StratifiedKFold(n_splits=cfg.FOLDS, shuffle=True, random_state=cfg.SEED)
    train["fold"] = -1
    for fold, (_, val_idx) in enumerate(skf.split(train, train["primary_label"])):
        train.loc[val_idx, "fold"] = fold

    print(f"Train clips     : {len(train):,}")
    print(f"Soundscape segs : {len(sc_df):,}")
    print(f"Classes         : {num_classes}")
    return train, sc_df, all_species, num_classes

Допоміжна функція для перетворення різних форматів часу в секунди:

In [31]:
def parse_time(time_val):
    if time_val is None or pd.isna(time_val):
        return 0.0
    
    if isinstance(time_val, str):
        if ':' in time_val:
            parts = time_val.strip().split(':')
            parts = [float(p) for p in parts]
            if len(parts) == 2:
                return parts[0] * 60 + parts[1]
            elif len(parts) == 3:
                return parts[0] * 3600 + parts[1] * 60 + parts[2]
        return float(time_val)
    
    return float(time_val)


def load_clip(filepath, cfg, start=None):
    offset = parse_time(start)
    
    y, _   = librosa.load(filepath, sr=cfg.SR, offset=offset, duration=cfg.DURATION)
    target = cfg.SR * cfg.DURATION
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    return y[:target].astype(np.float32)



def compute_melspec(y, cfg):
    mel    = librosa.feature.melspectrogram(
        y=y, sr=cfg.SR, n_fft=cfg.N_FFT, hop_length=cfg.HOP_LENGTH,
        n_mels=cfg.N_MELS, fmin=cfg.FMIN, fmax=cfg.FMAX,
    )
    mel_db = librosa.power_to_db(mel, ref=1.0).astype(np.float32)
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)
    return mel_db


def compute_pcen(y, cfg):
    S      = np.abs(librosa.stft(y, n_fft=cfg.N_FFT, hop_length=cfg.HOP_LENGTH)) ** 2
    mel_fb = librosa.filters.mel(sr=cfg.SR, n_fft=cfg.N_FFT, n_mels=cfg.N_MELS,
                                  fmin=cfg.FMIN, fmax=cfg.FMAX)
    mel    = mel_fb @ S
    pcen   = librosa.pcen(mel * (2**31), sr=cfg.SR,
                           hop_length=cfg.HOP_LENGTH).astype(np.float32)
    return (pcen - pcen.mean()) / (pcen.std() + 1e-6)


def spec_augment(mel, cfg):
    mel = mel.copy()
    for _ in range(cfg.NUM_MASKS):
        f  = random.randint(0, cfg.FREQ_MASK)
        f0 = random.randint(0, mel.shape[0] - f)
        mel[f0:f0+f, :] = 0.0
        t  = random.randint(0, cfg.TIME_MASK)
        t0 = random.randint(0, mel.shape[1] - t)
        mel[:, t0:t0+t] = 0.0
    return mel


In [32]:
class BirdDataset(Dataset):
    def __init__(self, df, cfg, augment=False):
        self.df      = df.reset_index(drop=True)
        self.cfg     = cfg
        self.augment = augment

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        y   = load_clip(row["filepath"], self.cfg, row.get("start", None))

        if self.augment:
            if random.random() < 0.5:
                y = np.roll(y, random.randint(-self.cfg.SR, self.cfg.SR))
            if random.random() < 0.3:
                y += np.random.normal(0, 0.005, y.shape).astype(np.float32)

        feat = compute_pcen(y, self.cfg) if self.cfg.USE_PCEN else compute_melspec(y, self.cfg)

        if self.augment and self.cfg.SPEC_AUG:
            feat = spec_augment(feat, self.cfg)

        mel   = torch.from_numpy(np.stack([feat, feat, feat]))
        label = torch.from_numpy(row["label_vec"])
        return mel, label

MixUp:

In [33]:
def mixup_data(x, y, alpha):
    lam   = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx   = torch.randperm(x.size(0), device=x.device)
    mixed = lam * x + (1 - lam) * x[idx]
    return mixed, y, y[idx], lam


def mixup_loss(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

Model:

In [34]:
class BirdModel(nn.Module):
    def __init__(self, cfg, num_classes):
        super().__init__()
        self.backbone = timm.create_model(
            cfg.MODEL, pretrained=cfg.PRETRAINED,
            in_chans=3, num_classes=0, global_pool="avg"
        )
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(self.backbone.num_features, num_classes),
        )
    def forward(self, x): return self.head(self.backbone(x))

Metric:

In [36]:
def macro_roc_auc(y_true, y_pred):
    aucs = []
    for i in range(y_true.shape[1]):
        if y_true[:, i].sum() == 0:
            continue
        try:
            aucs.append(roc_auc_score(y_true[:, i], y_pred[:, i]))
        except Exception:
            pass
    return float(np.mean(aucs)) if aucs else 0.0

Тренування:

In [37]:
def train_fold(cfg, train_df, sc_df, num_classes):
    fold  = cfg.TRAIN_FOLD
    trn_df = train_df[train_df["fold"] != fold].copy()
    val_df = train_df[train_df["fold"] == fold].copy()

    if len(sc_df) > 0:
        trn_df = pd.concat([trn_df, sc_df], ignore_index=True)

    print(f"\n{'='*60}")
    print(f"  {cfg.TAG.upper()} | Model: {cfg.MODEL} | Fold: {fold}")
    print(f"  MixUp: {cfg.MIXUP_ALPHA}  SpecAug: {cfg.SPEC_AUG}  PCEN: {cfg.USE_PCEN}")
    print(f"{'='*60}")
    print(f"  Train: {len(trn_df):,}  |  Val: {len(val_df):,}")

    trn_dl = DataLoader(BirdDataset(trn_df, cfg, augment=True),
                        batch_size=cfg.BATCH_SIZE, shuffle=True,
                        num_workers=cfg.NUM_WORKERS, pin_memory=True, drop_last=True)
    val_dl = DataLoader(BirdDataset(val_df, cfg, augment=False),
                        batch_size=cfg.BATCH_SIZE, shuffle=False,
                        num_workers=cfg.NUM_WORKERS, pin_memory=True)

    model  = BirdModel(cfg, num_classes).to(cfg.DEVICE)
    optim  = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=cfg.EPOCHS, eta_min=1e-6)
    crit   = nn.BCEWithLogitsLoss()
    scaler = GradScaler(enabled=cfg.AMP)

    best_auc  = 0.0
    save_path = cfg.OUTPUT_DIR / f"{cfg.TAG}_fold{fold}.pth"

    for epoch in range(1, cfg.EPOCHS + 1):
        model.train()
        run_loss = 0.0
        t0 = time.time()

        for mels, lbls in trn_dl:
            mels, lbls = mels.to(cfg.DEVICE), lbls.to(cfg.DEVICE)

            if cfg.MIXUP_ALPHA > 0:
                mels, y_a, y_b, lam = mixup_data(mels, lbls, cfg.MIXUP_ALPHA)
                with autocast(enabled=cfg.AMP):
                    loss = mixup_loss(crit, model(mels), y_a, y_b, lam)
            else:
                with autocast(enabled=cfg.AMP):
                    loss = crit(model(mels), lbls)

            optim.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optim); scaler.update()
            run_loss += loss.item()

        model.eval()
        preds_all, labels_all = [], []
        with torch.no_grad():
            for mels, lbls in val_dl:
                preds_all.append(torch.sigmoid(model(mels.to(cfg.DEVICE))).cpu().numpy())
                labels_all.append(lbls.numpy())

        auc = macro_roc_auc(np.concatenate(labels_all), np.concatenate(preds_all))
        sched.step()
        print(f"  Epoch {epoch:2d}/{cfg.EPOCHS} | "
              f"Loss: {run_loss/len(trn_dl):.4f} | "
              f"Val AUC: {auc:.4f} | {time.time()-t0:.0f}s")

        if auc > best_auc:
            best_auc = auc
            torch.save(model.state_dict(), save_path)
            print(f" Saved to {save_path.name}  (AUC={best_auc:.4f})")

    print(f"\n  Best Val AUC: {best_auc:.4f}")
    print(f"  Weights: {save_path}")
    del model; gc.collect(); torch.cuda.empty_cache()
    return best_auc

Модель 1:

In [1]:
─class CFG:
    BASE_DIR       = Path("/kaggle/input/competitions/birdclef-2026")
    TRAIN_CSV      = BASE_DIR / "train.csv"
    TAXON_CSV      = BASE_DIR / "taxonomy.csv"
    LABELS_CSV     = BASE_DIR / "train_soundscapes_labels.csv"
    AUDIO_DIR      = BASE_DIR / "train_audio"
    SOUNDSCAPE_DIR = BASE_DIR / "train_soundscapes"
    OUTPUT_DIR     = Path("/kaggle/working")

    SR             = 32_000
    DURATION       = 5
    N_FFT          = 1024
    HOP_LENGTH     = 512
    N_MELS         = 128
    FMIN           = 50
    FMAX           = 14_000

    SEED           = 42
    FOLDS          = 5
    TRAIN_FOLD     = 0  
    EPOCHS         = 10
    BATCH_SIZE     = 32
    LR             = 1e-3
    WEIGHT_DECAY   = 1e-4
    NUM_WORKERS    = 2
    AMP            = True
    MODEL          = "efficientnet_b0"
    PRETRAINED     = True
    DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"


def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(CFG.SEED)
print(f"Device: {CFG.DEVICE}")

def load_and_prepare():
    train    = pd.read_csv(CFG.TRAIN_CSV)
    taxonomy = pd.read_csv(CFG.TAXON_CSV)
    labels   = pd.read_csv(CFG.LABELS_CSV)

    all_species = sorted(taxonomy["primary_label"].tolist())
    sp2idx      = {sp: i for i, sp in enumerate(all_species)}
    NUM_CLASSES = len(all_species)

    def encode_labels(row):
        vec = np.zeros(NUM_CLASSES, dtype=np.float32)
        for sp in str(row["primary_label"]).split(";"):
            if sp in sp2idx:
                vec[sp2idx[sp]] = 1.0
        sec = str(row.get("secondary_labels", "")).strip("[]").replace("'", "")
        for sp in sec.split(", "):
            if sp.strip() in sp2idx:
                vec[sp2idx[sp.strip()]] = 0.5
        return vec

    train["label_vec"] = train.apply(encode_labels, axis=1)
    train["filepath"]  = train["filename"].apply(lambda f: str(CFG.AUDIO_DIR / f))

    sc_rows = []
    for _, r in labels.iterrows():
        vec = np.zeros(NUM_CLASSES, dtype=np.float32)
        for sp in str(r["primary_label"]).split(";"):
            sp = sp.strip()
            if sp in sp2idx:
                vec[sp2idx[sp]] = 1.0
        fpath = CFG.SOUNDSCAPE_DIR / r["filename"]
        if fpath.exists():
            sc_rows.append({
                "filepath":     str(fpath),
                "start":        r["start"],
                "label_vec":    vec,
                "primary_label": r["primary_label"],
            })
    sc_df = pd.DataFrame(sc_rows)

    skf = StratifiedKFold(n_splits=CFG.FOLDS, shuffle=True, random_state=CFG.SEED)
    train["fold"] = -1
    for fold, (_, val_idx) in enumerate(skf.split(train, train["primary_label"])):
        train.loc[val_idx, "fold"] = fold

    print(f"Train clips     : {len(train):,}")
    print(f"Soundscape segs : {len(sc_df):,}")
    print(f"Classes         : {NUM_CLASSES}")
    return train, sc_df, all_species, NUM_CLASSES

def parse_time(time_val):
    if time_val is None or pd.isna(time_val):
        return 0.0
    
    if isinstance(time_val, str):
        if ':' in time_val:
            parts = time_val.strip().split(':')
            parts = [float(p) for p in parts]
            if len(parts) == 2:
                return parts[0] * 60 + parts[1]
            elif len(parts) == 3: 
                return parts[0] * 3600 + parts[1] * 60 + parts[2]
        return float(time_val)
    
    return float(time_val)

def load_clip(filepath, start=None):
    offset = parse_time(start)
    y, _ = librosa.load(filepath, sr=CFG.SR, offset=offset, duration=CFG.DURATION)
    target = CFG.SR * CFG.DURATION
    if len(y) < target:
        y = np.pad(y, (0, int(target - len(y))))
        
    return y[:int(target)].astype(np.float32)

def compute_melspec(y):
    mel    = librosa.feature.melspectrogram(
        y=y, sr=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH,
        n_mels=CFG.N_MELS, fmin=CFG.FMIN, fmax=CFG.FMAX,
    )
    mel_db = librosa.power_to_db(mel, ref=1.0).astype(np.float32)
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)
    return mel_db


class BirdDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df      = df.reset_index(drop=True)
        self.augment = augment

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        y     = load_clip(row["filepath"], row.get("start", None))

        if self.augment:
            if random.random() < 0.5:
                y = np.roll(y, random.randint(-CFG.SR, CFG.SR))
            if random.random() < 0.3:
                y += np.random.normal(0, 0.005, y.shape).astype(np.float32)

        mel    = compute_melspec(y)
        mel    = torch.from_numpy(np.stack([mel, mel, mel]))
        label  = torch.from_numpy(row["label_vec"])
        return mel, label

class BirdModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = timm.create_model(
            CFG.MODEL, pretrained=CFG.PRETRAINED,
            in_chans=3, num_classes=0, global_pool="avg"
        )
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(self.backbone.num_features, num_classes),
        )
    def forward(self, x): return self.head(self.backbone(x))


def macro_roc_auc(y_true, y_pred):
    aucs = []
    for i in range(y_true.shape[1]):
        if y_true[:, i].sum() == 0:
            continue
        try:
            aucs.append(roc_auc_score(y_true[:, i], y_pred[:, i]))
        except Exception:
            pass
    return float(np.mean(aucs)) if aucs else 0.0

def train_fold(train_df, sc_df, num_classes, fold=0):
    trn_df = train_df[train_df["fold"] != fold].copy()
    val_df = train_df[train_df["fold"] == fold].copy()

    if len(sc_df) > 0:
        trn_df = pd.concat([trn_df, sc_df], ignore_index=True)

    print(f"Train: {len(trn_df):,}  |  Val: {len(val_df):,}")

    trn_dl = DataLoader(BirdDataset(trn_df, augment=True),
                        batch_size=CFG.BATCH_SIZE, shuffle=True,
                        num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=True)
    val_dl = DataLoader(BirdDataset(val_df, augment=False),
                        batch_size=CFG.BATCH_SIZE, shuffle=False,
                        num_workers=CFG.NUM_WORKERS, pin_memory=True)

    model  = BirdModel(num_classes).to(CFG.DEVICE)
    optim  = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=CFG.EPOCHS, eta_min=1e-6)
    crit   = nn.BCEWithLogitsLoss()
    scaler = GradScaler(enabled=CFG.AMP)

    best_auc  = 0.0
    save_path = CFG.OUTPUT_DIR / "approach1_effb0_fold0.pth"

    for epoch in range(1, CFG.EPOCHS + 1):
        model.train()
        run_loss = 0.0
        t0 = time.time()
        for mels, lbls in trn_dl:
            mels, lbls = mels.to(CFG.DEVICE), lbls.to(CFG.DEVICE)
            with autocast(enabled=CFG.AMP):
                loss = crit(model(mels), lbls)
            optim.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optim); scaler.update()
            run_loss += loss.item()

        model.eval()
        preds_all, labels_all = [], []
        with torch.no_grad():
            for mels, lbls in val_dl:
                preds_all.append(torch.sigmoid(model(mels.to(CFG.DEVICE))).cpu().numpy())
                labels_all.append(lbls.numpy())

        auc = macro_roc_auc(np.concatenate(labels_all), np.concatenate(preds_all))
        sched.step()
        print(f"Epoch {epoch:2d}/{CFG.EPOCHS} | "
              f"Loss: {run_loss/len(trn_dl):.4f} | "
              f"Val AUC: {auc:.4f} | "
              f"{time.time()-t0:.0f}s")

        if auc > best_auc:
            best_auc = auc
            torch.save(model.state_dict(), save_path)
            print(f"  ✓ Saved → {save_path.name}  (AUC={best_auc:.4f})")

    print(f"\nBest Val AUC: {best_auc:.4f}")
    print(f"Weights saved: {save_path}")
    del model; gc.collect(); torch.cuda.empty_cache()
    return best_auc


train_df, sc_df, all_species, NUM_CLASSES = load_and_prepare()
best_auc = train_fold(train_df, sc_df, NUM_CLASSES, fold=CFG.TRAIN_FOLD)

pd.DataFrame([{
    "approach": "EfficientNet-B0 baseline",
    "fold":     CFG.TRAIN_FOLD,
    "val_auc":  best_auc,
}]).to_csv(CFG.OUTPUT_DIR / "validation_results.csv", index=False)

print("\nDone ✓  →  approach1_effb0_fold0.pth + validation_results.csv")

Device: cuda
Train clips     : 35,549
Soundscape segs : 1,478
Classes         : 234
Train: 29,917  |  Val: 7,110


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Epoch  1/10 | Loss: 0.0296 | Val AUC: 0.8781 | 1272s
  ✓ Saved → approach1_effb0_fold0.pth  (AUC=0.8781)
Epoch  2/10 | Loss: 0.0181 | Val AUC: 0.9418 | 1067s
  ✓ Saved → approach1_effb0_fold0.pth  (AUC=0.9418)
Epoch  3/10 | Loss: 0.0144 | Val AUC: 0.9612 | 1083s
  ✓ Saved → approach1_effb0_fold0.pth  (AUC=0.9612)
Epoch  4/10 | Loss: 0.0121 | Val AUC: 0.9671 | 1094s
  ✓ Saved → approach1_effb0_fold0.pth  (AUC=0.9671)
Epoch  5/10 | Loss: 0.0103 | Val AUC: 0.9743 | 1077s
  ✓ Saved → approach1_effb0_fold0.pth  (AUC=0.9743)
Epoch  6/10 | Loss: 0.0088 | Val AUC: 0.9799 | 1075s
  ✓ Saved → approach1_effb0_fold0.pth  (AUC=0.9799)
Epoch  7/10 | Loss: 0.0074 | Val AUC: 0.9748 | 1080s
Epoch  8/10 | Loss: 0.0062 | Val AUC: 0.9773 | 1073s
Epoch  9/10 | Loss: 0.0053 | Val AUC: 0.9763 | 1086s
Epoch 10/10 | Loss: 0.0049 | Val AUC: 0.9754 | 1088s

Best Val AUC: 0.9799
Weights saved: /kaggle/working/approach1_effb0_fold0.pth

Done ✓  →  approach1_effb0_fold0.pth + validation_results.csv


In [ ]:
class CFG(BaseCFG):
    MODEL       = "efficientnet_b0"
    TAG         = "approach1_effb0"
    EPOCHS      = 10
    MIXUP_ALPHA = 0.0
    SPEC_AUG    = False
    USE_PCEN    = False
    
cfg = CFG()
seed_everything(cfg)

train_df, sc_df, all_species, num_classes = load_and_prepare(cfg)
best_auc = train_fold(cfg, train_df, sc_df, num_classes)

pd.DataFrame([{
    "approach": cfg.TAG,
    "model":    cfg.MODEL,
    "mixup":    cfg.MIXUP_ALPHA,
    "spec_aug": cfg.SPEC_AUG,
    "pcen":     cfg.USE_PCEN,
    "val_auc":  best_auc,
}]).to_csv(cfg.OUTPUT_DIR / f"results_{cfg.TAG}.csv", index=False)

Train clips     : 35,549
Soundscape segs : 1,478
Classes         : 234

  APPROACH1_EFFB0 | Model: efficientnet_b0 | Fold: 0
  MixUp: 0.0  SpecAug: False  PCEN: False
  Train: 29,917  |  Val: 7,110


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

  Epoch  1/10 | Loss: 0.0282 | Val AUC: 0.8767 | 963s
 Saved to approach1_effb0_fold0.pth  (AUC=0.8767)
  Epoch  2/10 | Loss: 0.0167 | Val AUC: 0.9441 | 873s
 Saved to approach1_effb0_fold0.pth  (AUC=0.9441)
  Epoch  3/10 | Loss: 0.0135 | Val AUC: 0.9673 | 855s
 Saved to approach1_effb0_fold0.pth  (AUC=0.9673)
  Epoch  4/10 | Loss: 0.0115 | Val AUC: 0.9646 | 842s
  Epoch  5/10 | Loss: 0.0099 | Val AUC: 0.9717 | 852s
 Saved to approach1_effb0_fold0.pth  (AUC=0.9717)
  Epoch  6/10 | Loss: 0.0084 | Val AUC: 0.9699 | 854s
  Epoch  7/10 | Loss: 0.0070 | Val AUC: 0.9718 | 912s
 Saved to approach1_effb0_fold0.pth  (AUC=0.9718)


### Модель 2

In [87]:
class CFG(BaseCFG):
    MODEL       = "efficientnet_b2"
    TAG         = "approach2_effb2_mixup"
    EPOCHS      = 10

    MIXUP_ALPHA = 0.5
    SPEC_AUG    = True
    FREQ_MASK   = 20
    TIME_MASK   = 40
    NUM_MASKS   = 2
    USE_PCEN    = False

In [88]:
cfg = CFG()
seed_everything(cfg)
print(f"Device: {cfg.DEVICE}")

train_df, sc_df, all_species, num_classes = load_and_prepare(cfg)
best_auc = train_fold(cfg, train_df, sc_df, num_classes)

pd.DataFrame([{
    "approach": cfg.TAG,
    "model":    cfg.MODEL,
    "mixup":    cfg.MIXUP_ALPHA,
    "spec_aug": cfg.SPEC_AUG,
    "pcen":     cfg.USE_PCEN,
    "val_auc":  best_auc,
}]).to_csv(cfg.OUTPUT_DIR / f"results_{cfg.TAG}.csv", index=False)

Device: cuda
Train clips     : 35,549
Soundscape segs : 1,478
Classes         : 234

  APPROACH2_EFFB2_MIXUP | Model: efficientnet_b2 | Fold: 0
  MixUp: 0.5  SpecAug: True  PCEN: False
  Train: 29,917  |  Val: 7,110
  Epoch  1/10 | Loss: 0.0323 | Val AUC: 0.8574 | 999s
 Saved to approach2_effb2_mixup_fold0.pth  (AUC=0.8574)
  Epoch  2/10 | Loss: 0.0243 | Val AUC: 0.9293 | 861s
 Saved to approach2_effb2_mixup_fold0.pth  (AUC=0.9293)
  Epoch  3/10 | Loss: 0.0215 | Val AUC: 0.9575 | 844s
 Saved to approach2_effb2_mixup_fold0.pth  (AUC=0.9575)
  Epoch  4/10 | Loss: 0.0196 | Val AUC: 0.9744 | 839s
 Saved to approach2_effb2_mixup_fold0.pth  (AUC=0.9744)
  Epoch  5/10 | Loss: 0.0183 | Val AUC: 0.9693 | 843s
  Epoch  6/10 | Loss: 0.0175 | Val AUC: 0.9729 | 864s
  Epoch  7/10 | Loss: 0.0165 | Val AUC: 0.9803 | 854s
 Saved to approach2_effb2_mixup_fold0.pth  (AUC=0.9803)
  Epoch  8/10 | Loss: 0.0158 | Val AUC: 0.9784 | 853s
  Epoch  9/10 | Loss: 0.0150 | Val AUC: 0.9809 | 856s
 Saved to approach

### Модель 3

In [15]:
class CFG(BaseCFG):
    MODEL       = "efficientnet_b1"
    TAG         = "approach3_pcen"
    EPOCHS      = 10

    MIXUP_ALPHA = 0.3
    SPEC_AUG    = True
    FREQ_MASK   = 15
    TIME_MASK   = 30
    NUM_MASKS   = 2
    USE_PCEN    = True

cfg = CFG()
seed_everything(cfg)

train_df, sc_df, all_species, num_classes = load_and_prepare(cfg)
best_auc = train_fold(cfg, train_df, sc_df, num_classes)

pd.DataFrame([{
    "approach": cfg.TAG,
    "model":    cfg.MODEL,
    "mixup":    cfg.MIXUP_ALPHA,
    "spec_aug": cfg.SPEC_AUG,
    "pcen":     cfg.USE_PCEN,
    "val_auc":  best_auc,
}]).to_csv(cfg.OUTPUT_DIR / f"results_{cfg.TAG}.csv", index=False)

Train clips     : 35,549
Soundscape segs : 1,478
Classes         : 234

  APPROACH3_PCEN | Model: efficientnet_b1 | Fold: 0
  MixUp: 0.3  SpecAug: True  PCEN: True
  Train: 29,917  |  Val: 7,110


model.safetensors:   0%|          | 0.00/31.5M [00:00<?, ?B/s]

  Epoch  1/10 | Loss: 0.0331 | Val AUC: 0.7034 | 1417s
 Saved to approach3_pcen_fold0.pth  (AUC=0.7034)
  Epoch  2/10 | Loss: 0.0281 | Val AUC: 0.7923 | 944s
 Saved to approach3_pcen_fold0.pth  (AUC=0.7923)
  Epoch  3/10 | Loss: 0.0264 | Val AUC: 0.8555 | 927s
 Saved to approach3_pcen_fold0.pth  (AUC=0.8555)
  Epoch  4/10 | Loss: 0.0242 | Val AUC: 0.9137 | 953s
 Saved to approach3_pcen_fold0.pth  (AUC=0.9137)
  Epoch  5/10 | Loss: 0.0226 | Val AUC: 0.9154 | 955s
 Saved to approach3_pcen_fold0.pth  (AUC=0.9154)
  Epoch  6/10 | Loss: 0.0213 | Val AUC: 0.9318 | 940s
 Saved to approach3_pcen_fold0.pth  (AUC=0.9318)
  Epoch  7/10 | Loss: 0.0201 | Val AUC: 0.9508 | 928s
 Saved to approach3_pcen_fold0.pth  (AUC=0.9508)
  Epoch  8/10 | Loss: 0.0190 | Val AUC: 0.9477 | 917s
  Epoch  9/10 | Loss: 0.0183 | Val AUC: 0.9485 | 933s
  Epoch 10/10 | Loss: 0.0179 | Val AUC: 0.9549 | 938s
 Saved to approach3_pcen_fold0.pth  (AUC=0.9549)

  Best Val AUC: 0.9549
  Weights: /kaggle/working/approach3_pcen_f

### Модель 4
ConvNeXt-Small | MixUp | SpecAugment | Retrain on 100%

In [23]:
def _one_epoch(model, loader, cfg, crit, scaler, optim=None):
    is_train = optim is not None
    model.train() if is_train else model.eval()
    run_loss = 0.0
    preds_all, labels_all = [], []
 
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for mels, lbls in loader:
            mels, lbls = mels.to(cfg.DEVICE), lbls.to(cfg.DEVICE)
            if is_train:
                if cfg.MIXUP_ALPHA > 0:
                    mels, y_a, y_b, lam = mixup_data(mels, lbls, cfg.MIXUP_ALPHA)
                    with autocast(enabled=cfg.AMP):
                        loss = mixup_loss(crit, model(mels), y_a, y_b, lam)
                else:
                    with autocast(enabled=cfg.AMP):
                        loss = crit(model(mels), lbls)
                optim.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optim); scaler.update()
                run_loss += loss.item()
            else:
                preds_all.append(torch.sigmoid(model(mels)).cpu().numpy())
                labels_all.append(lbls.cpu().numpy())
 
    if is_train:
        return run_loss / len(loader)
    return np.concatenate(preds_all), np.concatenate(labels_all)
    
def train_fold(cfg, train_df, sc_df, num_classes):
    fold   = cfg.TRAIN_FOLD
    trn_df = train_df[train_df["fold"] != fold].copy()
    val_df = train_df[train_df["fold"] == fold].copy()
    if len(sc_df) > 0:
        trn_df = pd.concat([trn_df, sc_df], ignore_index=True)
 
    print(f"\n{'='*60}")
    print(f"  {cfg.TAG.upper()} | {cfg.MODEL} | Fold {fold}")
    print(f"  MixUp={cfg.MIXUP_ALPHA}  SpecAug={cfg.SPEC_AUG}  PCEN={cfg.USE_PCEN}")
    print(f"{'='*60}")
    print(f"  Train: {len(trn_df):,}  |  Val: {len(val_df):,}")
 
    trn_dl = DataLoader(BirdDataset(trn_df, cfg, augment=True),
                        batch_size=cfg.BATCH_SIZE, shuffle=True,
                        num_workers=cfg.NUM_WORKERS, pin_memory=True, drop_last=True)
    val_dl = DataLoader(BirdDataset(val_df, cfg, augment=False),
                        batch_size=cfg.BATCH_SIZE, shuffle=False,
                        num_workers=cfg.NUM_WORKERS, pin_memory=True)
 
    model  = BirdModel(cfg, num_classes).to(cfg.DEVICE)
    optim  = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=cfg.EPOCHS, eta_min=1e-6)
    crit   = nn.BCEWithLogitsLoss()
    scaler = GradScaler(enabled=cfg.AMP)
 
    best_auc, best_epoch = 0.0, 1
    save_path = cfg.OUTPUT_DIR / f"{cfg.TAG}_fold{fold}.pth"
 
    for epoch in range(1, cfg.EPOCHS + 1):
        t0   = time.time()
        loss = _one_epoch(model, trn_dl, cfg, crit, scaler, optim)
        y_pred, y_true = _one_epoch(model, val_dl, cfg, crit, scaler, optim=None)
        auc  = macro_roc_auc(y_true, y_pred)
        sched.step()
 
        marker = ""
        if auc > best_auc:
            best_auc, best_epoch = auc, epoch
            torch.save(model.state_dict(), save_path)
            marker = f"  ✓ best"
 
        print(f"  Epoch {epoch:2d}/{cfg.EPOCHS} | "
              f"Loss={loss:.4f} | AUC={auc:.4f} | "
              f"{time.time()-t0:.0f}s{marker}")
 
    print(f"\n  Best Val AUC={best_auc:.4f} @ epoch {best_epoch}")
    del model; gc.collect(); torch.cuda.empty_cache()
    return best_auc, best_epoch

def retrain_full_data(cfg, train_df, sc_df, num_classes, best_epoch):
    full_df = train_df.copy()
    if len(sc_df) > 0:
        full_df = pd.concat([full_df, sc_df], ignore_index=True)
 
    print(f"\n{'='*60}")
    print(f"  RETRAIN ON 100% DATA | {cfg.TAG} | epochs={best_epoch}")
    print(f"{'='*60}")
    print(f"  Total samples: {len(full_df):,}")
 
    full_dl = DataLoader(BirdDataset(full_df, cfg, augment=True),
                         batch_size=cfg.BATCH_SIZE, shuffle=True,
                         num_workers=cfg.NUM_WORKERS, pin_memory=True, drop_last=True)
 
    model  = BirdModel(cfg, num_classes).to(cfg.DEVICE)
    optim  = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=best_epoch, eta_min=1e-6)
    crit   = nn.BCEWithLogitsLoss()
    scaler = GradScaler(enabled=cfg.AMP)
 
    save_path = cfg.OUTPUT_DIR / f"{cfg.TAG}_full.pth"
 
    for epoch in range(1, best_epoch + 1):
        t0   = time.time()
        loss = _one_epoch(model, full_dl, cfg, crit, scaler, optim)
        sched.step()
        print(f"  Epoch {epoch:2d}/{best_epoch} | Loss={loss:.4f} | {time.time()-t0:.0f}s")
 
    torch.save(model.state_dict(), save_path)
    print(f"\n  ✓ Full retrain done → {save_path.name}")
    del model; gc.collect(); torch.cuda.empty_cache()
    return save_path

In [24]:
class CFG(BaseCFG):
    MODEL        = "convnext_small"
    TAG          = "approach5_convnext_small"
    EPOCHS       = 12

    MIXUP_ALPHA  = 0.4
    SPEC_AUG     = True
    FREQ_MASK    = 20
    TIME_MASK    = 40
    NUM_MASKS    = 2
    USE_PCEN     = False

    LR           = 4e-4
    BATCH_SIZE   = 24
    WEIGHT_DECAY = 1e-2

cfg = CFG()
seed_everything(cfg)
print(f"Device: {cfg.DEVICE}")
print(f"Model : {cfg.MODEL}  (ConvNeXt-Small, ~50M params)")

train_df, sc_df, all_species, num_classes = load_and_prepare(cfg)

best_auc, best_epoch = train_fold(cfg, train_df, sc_df, num_classes)

print(f"\n  Val AUC = {best_auc:.4f}  @ epoch {best_epoch}")
print(f" Запускаємо retrain на 100% даних ({best_epoch} epochs)...")

full_path = retrain_full_data(cfg, train_df, sc_df, num_classes, best_epoch)

pd.DataFrame([{
    "approach":    cfg.TAG,
    "model":       cfg.MODEL,
    "best_epoch":  best_epoch,
    "val_auc":     best_auc,
    "full_weights": str(full_path),
}]).to_csv(cfg.OUTPUT_DIR / f"results_{cfg.TAG}.csv", index=False)

print(f"""
  Val weights  : {cfg.TAG}_fold{cfg.TRAIN_FOLD}.pth  (для аналізу метрик)
  Full weights : {cfg.TAG}_full.pth              (для submission)
""")

Device: cuda
Model : convnext_small  (ConvNeXt-Small, ~50M params)
Train clips     : 35,549
Soundscape segs : 1,478
Classes         : 234

  APPROACH5_CONVNEXT_SMALL | convnext_small | Fold 0
  MixUp=0.4  SpecAug=True  PCEN=False
  Train: 29,917  |  Val: 7,110
  Epoch  1/12 | Loss=0.0349 | AUC=0.4708 | 1339s  ✓ best
  Epoch  2/12 | Loss=0.0319 | AUC=0.4568 | 982s
  Epoch  3/12 | Loss=0.0317 | AUC=0.4492 | 986s
  Epoch  4/12 | Loss=0.0317 | AUC=0.4626 | 1026s
  Epoch  5/12 | Loss=0.0316 | AUC=0.4799 | 1026s  ✓ best
  Epoch  6/12 | Loss=0.0315 | AUC=0.4777 | 1076s
  Epoch  7/12 | Loss=0.0315 | AUC=0.4697 | 1100s
  Epoch  8/12 | Loss=0.0314 | AUC=0.4843 | 1108s  ✓ best
  Epoch  9/12 | Loss=0.0314 | AUC=0.4855 | 1115s  ✓ best
  Epoch 10/12 | Loss=0.0313 | AUC=0.4814 | 1122s
  Epoch 11/12 | Loss=0.0313 | AUC=0.5023 | 1105s  ✓ best
  Epoch 12/12 | Loss=0.0313 | AUC=0.4936 | 1101s

  Best Val AUC=0.5023 @ epoch 11

  Val AUC = 0.5023  @ epoch 11
 Запускаємо retrain на 100% даних (11 epochs)..